# Regresión múltiple inferencial — Salsberry Realty

## Propósito

> "Salsberry Realty vende casas a lo largo de la costa este de los Estados Unidos. Una de las preguntas más frecuentes formuladas por los posibles compradores es la siguiente: Si compramos esta casa, ¿cuánto podemos esperar a pagar en calefacción durante el invierno? El departamento de investigación de Salsberry ha sido invitado a desarrollar algunas directrices relativas a los costos de calefacción para viviendas familias normales"

Este notebook desarrolla un modelo de **regresión lineal múltiple con Statsmodels** para explicar el costo de calefacción (`Cost`) mediante temperatura exterior (`Temp`), aislamiento (`Insul`), antigüedad (`Age`) y presencia de cochera (`Garage`). El énfasis es inferencial: estimar efectos parciales, intervalos de confianza, significancia estadística y verificar los supuestos del modelo.

> **Nota de interpretación:** una asociación estadística, aun controlando por otras variables, no demuestra causalidad. Las unidades monetarias y escalas se conservan tal como aparecen en el archivo fuente.

## 1. Configuración del entorno

Este bloque instala las bibliotecas requeridas. La opción `-q` reduce mensajes de instalación en Colab. Luego se importan herramientas para análisis, visualización, diagnóstico y la interfaz Gradio.

In [ ]:
!pip -q install statsmodels gradio seaborn

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.graphics.gofplots import qqplot
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
ALPHA = 0.05
print("Entorno preparado correctamente.")

## 2. Carga del archivo `Costocal.csv`

En Google Colab se abrirá un selector para subir el CSV. Fuera de Colab, el bloque busca el archivo en la carpeta actual y en `upload/`. Las filas totalmente vacías se eliminan; no se imputan ni modifican observaciones válidas.

In [ ]:
try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No se cargó ningún archivo.")
    csv_path = next(iter(uploaded))
else:
    candidates = ["Costocal.csv", "upload/Costocal.csv"]
    csv_path = next((p for p in candidates if os.path.exists(p)), None)
    if csv_path is None:
        raise FileNotFoundError("Coloque Costocal.csv junto al notebook o en upload/.")

datos_brutos = pd.read_csv(csv_path)
datos = datos_brutos.dropna(how="all").copy()

columnas = ["Cost", "Temp", "Insul", "Age", "Garage"]
faltantes = [c for c in columnas if c not in datos.columns]
if faltantes:
    raise ValueError(f"Faltan columnas requeridas: {faltantes}")

datos = datos[columnas].apply(pd.to_numeric, errors="coerce").dropna()
datos["Garage"] = datos["Garage"].astype(int)

print(f"Filas originales: {len(datos_brutos):,}")
print(f"Observaciones válidas usadas: {len(datos):,}")
display(datos.head())

## 3. Diccionario y exploración inicial

- `Cost`: costo de calefacción invernal; variable dependiente **Y**.
- `Temp`: temperatura exterior; variable explicativa **X**.
- `Insul`: nivel de aislamiento; variable explicativa **X**.
- `Age`: antigüedad de la vivienda; variable explicativa **X**.
- `Garage`: 1 si tiene cochera y 0 si no; variable indicadora **X**.

Los descriptivos permiten revisar rangos, dispersión y posibles errores antes de modelar.

In [ ]:
display(datos.describe().T.round(2))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, variable in zip(axes.flat, columnas):
    if variable == "Garage":
        sns.countplot(data=datos, x=variable, ax=ax, color="#3B82F6")
    else:
        sns.histplot(data=datos, x=variable, kde=True, ax=ax, color="#2563EB")
    ax.set_title(f"Distribución de {variable}")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

## 4. Relaciones bivariadas X–Y

Los gráficos muestran la relación *sin controlar* por las demás variables. Para las variables continuas se añade una recta de tendencia; para `Garage` se comparan las distribuciones del costo. La inferencia principal se realizará después con efectos parciales.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, x in zip(axes.flat[:3], ["Temp", "Insul", "Age"]):
    sns.regplot(data=datos, x=x, y="Cost", ci=95,
                scatter_kws={"s": 60, "alpha": .8},
                line_kws={"color": "#DC2626"}, ax=ax)
    ax.set_title(f"Cost frente a {x}")

sns.boxplot(data=datos, x="Garage", y="Cost", ax=axes.flat[3], color="#93C5FD")
sns.stripplot(data=datos, x="Garage", y="Cost", ax=axes.flat[3], color="#1E3A8A", size=7)
axes.flat[3].set_title("Cost según presencia de Garage")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(datos.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlaciones")
plt.tight_layout()
plt.show()

## 5. Especificación y ajuste del modelo

El modelo es:

$$Cost_i=\beta_0+\beta_1Temp_i+\beta_2Insul_i+\beta_3Age_i+\beta_4Garage_i+\varepsilon_i$$

Cada coeficiente representa el cambio esperado en `Cost` ante un incremento de una unidad en esa X, **manteniendo constantes las demás variables**. `Garage` compara viviendas con cochera frente a viviendas sin cochera.

In [ ]:
formula = "Cost ~ Temp + Insul + Age + Garage"
modelo = smf.ols(formula, data=datos).fit()
print(modelo.summary())

## 6. Tabla inferencial e interpretación automática

La tabla reúne coeficientes, errores estándar, estadísticos t, valores p e intervalos de confianza al 95%. Un resultado se considera estadísticamente significativo si `p < 0.05` y su intervalo no incluye cero.

In [ ]:
ic = modelo.conf_int(alpha=ALPHA)
tabla = pd.DataFrame({
    "Coeficiente": modelo.params,
    "Error estándar": modelo.bse,
    "t": modelo.tvalues,
    "p-valor": modelo.pvalues,
    "IC 95% inferior": ic[0],
    "IC 95% superior": ic[1],
})
tabla["Significativa (5%)"] = tabla["p-valor"] < ALPHA
display(tabla.round(4))

nombres = {
    "Temp": "temperatura",
    "Insul": "nivel de aislamiento",
    "Age": "antigüedad",
    "Garage": "presencia de cochera"
}

for var in ["Temp", "Insul", "Age", "Garage"]:
    b, p = modelo.params[var], modelo.pvalues[var]
    if var == "Garage":
        efecto = f"una vivienda con cochera presenta un Cost esperado {abs(b):.2f} unidades {'mayor' if b > 0 else 'menor'} que una sin cochera"
    else:
        efecto = f"una unidad adicional se asocia con {abs(b):.2f} unidades {'más' if b > 0 else 'menos'} de Cost esperado"
    decision = "relación significativa" if p < ALPHA else "evidencia insuficiente de relación al 5%"
    print(f"• {nombres[var].capitalize()}: {efecto}, controlando las demás X; p={p:.4f} ({decision}).")

print(f"\nR²={modelo.rsquared:.3f}; R² ajustado={modelo.rsquared_adj:.3f}; "
      f"F={modelo.fvalue:.2f}, p global={modelo.f_pvalue:.3g}.")

## 7. Impacto parcial de cada X sobre Y

Estos gráficos son más apropiados que una correlación simple para interpretar regresión múltiple: muestran la relación entre cada X y `Cost` después de retirar linealmente la influencia de las demás variables. La pendiente coincide con el signo del coeficiente parcial.

In [ ]:
fig = sm.graphics.plot_partregress_grid(modelo, fig=plt.figure(figsize=(13, 9)))
fig.suptitle("Gráficos de regresión parcial: impacto de cada X en Cost", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

coef = modelo.params.drop("Intercept")
err = 1.96 * modelo.bse.drop("Intercept")
colors = ["#16A34A" if modelo.pvalues[v] < ALPHA else "#9CA3AF" for v in coef.index]
plt.figure(figsize=(9, 5))
plt.errorbar(coef.index, coef.values, yerr=err.values, fmt="none", ecolor="#334155", capsize=5)
plt.scatter(coef.index, coef.values, c=colors, s=100, zorder=3)
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Coeficiente estimado (IC aproximado 95%)")
plt.title("Efectos parciales: verde = significativo al 5%")
plt.tight_layout()
plt.show()

## 8. Verificación de supuestos

Se evalúan: linealidad/especificación (Ramsey RESET), normalidad de residuos (Jarque–Bera y Q–Q), homocedasticidad (Breusch–Pagan), independencia aproximada (Durbin–Watson), ausencia de multicolinealidad severa (VIF) e influencia (Cook). **No rechazar una prueba no demuestra que el supuesto sea perfecto**, especialmente con solo 20 observaciones; indica que no se detectó evidencia suficiente de incumplimiento.

In [ ]:
residuos = modelo.resid
ajustados = modelo.fittedvalues

jb_stat, jb_p, skew, kurt = jarque_bera(residuos)
bp_lm, bp_p, bp_f, bp_f_p = het_breuschpagan(residuos, modelo.model.exog)
dw = durbin_watson(residuos)
reset = linear_reset(modelo, power=2, use_f=True)

X_vif = pd.DataFrame(modelo.model.exog, columns=modelo.model.exog_names)
vif = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})

influencia = modelo.get_influence()
cook = influencia.cooks_distance[0]
umbral_cook = 4 / len(datos)

diagnosticos = pd.DataFrame({
    "Diagnóstico": ["Jarque–Bera", "Breusch–Pagan", "Ramsey RESET", "Durbin–Watson"],
    "Estadístico": [jb_stat, bp_lm, float(reset.fvalue), dw],
    "p-valor": [jb_p, bp_p, float(reset.pvalue), np.nan],
    "Criterio orientativo": ["p > .05", "p > .05", "p > .05", "cercano a 2"]
})
display(diagnosticos.round(4))
display(vif.round(3))
print(f"Cook máximo = {cook.max():.3f}; umbral orientativo 4/n = {umbral_cook:.3f}; observación = {cook.argmax()+1}.")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.scatterplot(x=ajustados, y=residuos, ax=axes[0, 0], s=70)
axes[0, 0].axhline(0, color="#DC2626")
axes[0, 0].set(xlabel="Valores ajustados", ylabel="Residuos", title="Residuos vs. ajustados")

qqplot(residuos, line="45", fit=True, ax=axes[0, 1])
axes[0, 1].set_title("Q–Q de residuos")

sns.scatterplot(x=ajustados, y=np.sqrt(np.abs(influencia.resid_studentized_internal)),
                ax=axes[1, 0], s=70)
axes[1, 0].set(xlabel="Valores ajustados", ylabel="√|residuo estudentizado|", title="Scale–Location")

axes[1, 1].stem(np.arange(1, len(datos)+1), cook, basefmt=" ")
axes[1, 1].axhline(umbral_cook, color="#DC2626", linestyle="--", label="4/n")
axes[1, 1].set(xlabel="Observación", ylabel="Distancia de Cook", title="Influencia")
axes[1, 1].legend()
plt.tight_layout()
plt.show()

## 9. Sensibilidad con errores estándar robustos HC3

Aunque Breusch–Pagan no detecte heterocedasticidad, HC3 es una revisión prudente para muestras pequeñas. Los coeficientes no cambian; se recalculan errores estándar, intervalos y valores p. Si las conclusiones se mantienen, la inferencia es más estable.

In [ ]:
modelo_hc3 = modelo.get_robustcov_results(cov_type="HC3")
robusta = pd.DataFrame({
    "Variable": modelo.model.exog_names,
    "Coeficiente": modelo_hc3.params,
    "EE HC3": modelo_hc3.bse,
    "p-valor HC3": modelo_hc3.pvalues,
    "IC95% inferior HC3": modelo_hc3.conf_int()[:, 0],
    "IC95% superior HC3": modelo_hc3.conf_int()[:, 1],
})
display(robusta.round(4))

## 10. Predicción con intervalo

Para una vivienda nueva se reporta la estimación media y el intervalo de predicción al 95%. Este último es más amplio porque incorpora tanto la incertidumbre del modelo como la variabilidad individual de una vivienda.

In [ ]:
ejemplo = pd.DataFrame({"Temp": [35], "Insul": [6], "Age": [7], "Garage": [0]})
pred_ejemplo = modelo.get_prediction(ejemplo).summary_frame(alpha=0.05)
display(pd.concat([ejemplo, pred_ejemplo], axis=1).round(2))

## 11. Interfaz Gradio explicativa

Los controles permiten simular una vivienda. La salida presenta costo esperado, intervalo de predicción y una explicación de las relaciones estadísticamente significativas. En Colab, `share=True` crea un enlace temporal; la interfaz no sustituye una evaluación profesional y no debe usarse fuera de los rangos observados.

In [ ]:
import gradio as gr

rangos = datos.agg(["min", "max", "median"])

def explicar_vivienda(temp, insul, age, garage):
    nueva = pd.DataFrame({
        "Temp": [float(temp)], "Insul": [float(insul)],
        "Age": [float(age)], "Garage": [int(garage)]
    })
    pred = modelo.get_prediction(nueva).summary_frame(alpha=0.05).iloc[0]
    lineas = [
        f"## Costo esperado: **{pred['mean']:.2f} unidades monetarias**",
        f"**IC 95% de la media:** {pred['mean_ci_lower']:.2f} a {pred['mean_ci_upper']:.2f}",
        f"**Intervalo de predicción 95% para una vivienda:** {pred['obs_ci_lower']:.2f} a {pred['obs_ci_upper']:.2f}",
        "### Relaciones significativas del modelo"
    ]
    for var, etiqueta in nombres.items():
        if modelo.pvalues[var] < ALPHA:
            b = modelo.params[var]
            if var == "Garage":
                texto = f"Tener cochera se asocia con {abs(b):.2f} unidades {'más' if b > 0 else 'menos'} de costo, frente a no tenerla."
            else:
                texto = f"Cada unidad adicional de {etiqueta} se asocia con {abs(b):.2f} unidades {'más' if b > 0 else 'menos'} de costo."
            lineas.append(f"- **{etiqueta.capitalize()}**: {texto} (p={modelo.pvalues[var]:.4f}), manteniendo las demás variables constantes.")
    lineas.append("\n*Asociación no implica causalidad. Evite extrapolar fuera de los rangos de la muestra.*")
    return "\n".join(lineas)

demo = gr.Interface(
    fn=explicar_vivienda,
    inputs=[
        gr.Slider(float(rangos.loc["min", "Temp"]), float(rangos.loc["max", "Temp"]), value=float(rangos.loc["median", "Temp"]), label="Temperatura (Temp)"),
        gr.Slider(float(rangos.loc["min", "Insul"]), float(rangos.loc["max", "Insul"]), value=float(rangos.loc["median", "Insul"]), label="Aislamiento (Insul)"),
        gr.Slider(float(rangos.loc["min", "Age"]), float(rangos.loc["max", "Age"]), value=float(rangos.loc["median", "Age"]), label="Antigüedad (Age)"),
        gr.Radio([0, 1], value=0, label="Garage: 0=No, 1=Sí")
    ],
    outputs=gr.Markdown(),
    title="Salsberry Realty — simulador explicativo",
    description="Estime el costo de calefacción y conozca las relaciones significativas del modelo."
)

if EN_COLAB:
    demo.launch(share=True, debug=False)
else:
    print("Interfaz construida. En Colab se abrirá automáticamente al ejecutar esta celda.")
    display(Markdown(explicar_vivienda(35, 6, 7, 0)))

## 12. Conclusiones

Al ejecutar con `Costocal.csv`, el modelo explica aproximadamente **88.0%** de la variación de `Cost` (R² ajustado ≈ **84.8%**) y es globalmente significativo. Manteniendo constantes las demás variables:

- Una unidad adicional de `Temp` se asocia con cerca de **3.70 unidades menos** de costo; la relación es significativa.
- Una unidad adicional de `Insul` se asocia con cerca de **11.66 unidades menos** de costo; la relación es significativa.
- Tener `Garage` se asocia con cerca de **71.50 unidades más** de costo frente a no tenerla; la relación es significativa. Esto puede reflejar tamaño u otras características omitidas y no debe interpretarse automáticamente como efecto causal de la cochera.
- `Age` presenta un coeficiente positivo, pero **no es estadísticamente significativo al 5%**; con esta muestra no existe evidencia suficiente para afirmar una relación independiente.

Los diagnósticos no detectan evidencia estadística de no linealidad, heterocedasticidad o no normalidad; los VIF son bajos y Durbin–Watson se encuentra cerca de 2. Una observación queda apenas por encima del umbral orientativo de Cook `4/n`, por lo que conviene revisarla y ampliar la muestra. Con solo 20 casos y cuatro predictores, los resultados deben considerarse directrices preliminares, no reglas universales.

### Recomendación

Antes de desplegar el modelo, recopilar más viviendas e incorporar variables plausibles como superficie, combustible, eficiencia del sistema, ocupación, zona climática y precio local de energía. Esto permitiría validar el modelo fuera de muestra y reducir sesgo por variables omitidas.

## 13. Referencias metodológicas

- Statsmodels Developers. *Ordinary Least Squares (OLS)* y diagnósticos de regresión: https://www.statsmodels.org/
- Wooldridge, J. M. (2020). *Introductory Econometrics: A Modern Approach*. Cengage.
- Kutner, M. H., Nachtsheim, C. J., Neter, J., & Li, W. (2005). *Applied Linear Statistical Models*. McGraw-Hill.

**Reproducibilidad:** ejecute todas las celdas en orden. El notebook fija `α = 0.05`, conserva los datos originales válidos y limita los controles Gradio a los rangos observados.